# 🧬 pg_bio: Direct Genomics I/O (VCF Parsing)
Traditionally, processing genomic VCF (Variant Call Format) files requires heavy external Python pipelines or CLI tools like `bcftools`.

With **`pg_bio`**, we integrate the Rust `noodles` crate directly into PostgreSQL. This allows us to map VCF files natively on disk directly to Virtual Tables, eliminating parsing bottlenecks and memory overhead!

In [1]:
# Setup: Create a temporary VCF file
vcf_content = """##fileformat=VCFv4.2
##INFO=<ID=DP,Number=1,Type=Integer,Description="Total Depth">
##INFO=<ID=CLNSIG,Number=.,Type=String,Description="ClinVar significance">
#CHROM	POS	ID	REF	ALT	QUAL	FILTER	INFO
chr1	10000	rs123	A	T	50	PASS	DP=100;CLNSIG=Pathogenic
chr1	20000	rs456	G	C	99	PASS	DP=150
chr2	50000	rs789	C	G	20	LowQual	DP=10;CLNSIG=Benign
chr2	50500	rs101	T	A	85	PASS	DP=200;CLNSIG=Pathogenic
"""
with open('/tmp/tutorial.vcf', 'w') as f:
    f.write(vcf_content)


## 2. Natively Querying the VCF file with `pgbio-py`
We can execute a direct `SELECT` against the virtual table `parse_vcf('/tmp/tutorial.vcf')`!

In [2]:
import pandas as pd
from pgbio import PgBioClient

# Connect to our local pg_bio development instance
client = PgBioClient("postgresql://localhost:28818/pg_bio")

# Execute native parsing
variants = client.parse_vcf('/tmp/tutorial.vcf')
df = pd.DataFrame(variants)
df


,chrom,pos,id,ref,alt,qual,filter,info
0,chr1,10000,rs123,A,T,50.0,PASS,"DP=Integer(100);CLNSIG=Array(String([Some(""Pat..."
1,chr1,20000,rs456,G,C,99.0,PASS,DP=Integer(150)
2,chr2,50000,rs789,C,G,20.0,LowQual,"DP=Integer(10);CLNSIG=Array(String([Some(""Beni..."
3,chr2,50500,rs101,T,A,85.0,PASS,"DP=Integer(200);CLNSIG=Array(String([Some(""Pat..."


## 3. The Power of Direct SQL Integration
By wrapping `parse_vcf` in standard SQL, you can seamlessly integrate it with PostgreSQL's spatial `int4range` features. For instance, filtering pathogenic variants directly into a GiST-indexed table!

In [3]:
with client._get_conn() as conn:
    with conn.cursor() as cur:
        # Create a table using int4range for the genomic spatial coordinates
        cur.execute("""
            CREATE TEMP TABLE patient_variants (
                chrom TEXT,
                genomic_range int4range,
                id TEXT,
                info TEXT
            );
        """)
        
        # Directly inject Pathogenic variants from the VCF!
        cur.execute("""
            INSERT INTO patient_variants (chrom, genomic_range, id, info)
            SELECT chrom, int4range(pos, pos + length(alt_allele)), id, info
            FROM parse_vcf('/tmp/tutorial.vcf')
            WHERE info LIKE '%Pathogenic%';
        """)
        
        cur.execute("SELECT * FROM patient_variants;")
        print("Inserted Pathogenic Spatial Ranges:")
        for row in cur.fetchall():
            print(row)


Inserted Pathogenic Spatial Ranges:
('chr1', Range(10000, 10001, '[)'), 'rs123', 'DP=Integer(100);CLNSIG=Array(String([Some("Pathogenic")]))')
('chr2', Range(50500, 50501, '[)'), 'rs101', 'DP=Integer(200);CLNSIG=Array(String([Some("Pathogenic")]))')
